In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.datasets import MNIST
from torchvision import transforms

from torch.utils.data import DataLoader

In [2]:
print("PyTorch Version :", torch.__version__)

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU Name :", torch.cuda.get_device_name(0))

    print("CUDA Version :", torch.version.cuda)

PyTorch Version : 2.12.1+cu132
CUDA Available : True
GPU Name : NVIDIA GeForce RTX 3050 6GB Laptop GPU
CUDA Version : 13.2


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device :", device)

Using Device : cuda


In [4]:
transform = transforms.ToTensor()

train_dataset = MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

In [5]:
trainloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

testloader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

In [6]:
class MLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Flatten(),

            nn.Linear(784,256),
            nn.ReLU(),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Linear(128,10)

        )

    def forward(self,x):

        return self.network(x)

In [7]:
model = MLP().to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print(next(model.parameters()).device)

cuda:0


In [8]:
def train(model, trainloader, testloader, epochs):

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        for images, labels in trainloader:

            # Move Batch to GPU
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in testloader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _, pred = torch.max(outputs,1)

                total += labels.size(0)

                correct += (pred==labels).sum().item()

        print(
            f"Epoch {epoch+1}/{epochs}"
            f"  Loss={running_loss/len(trainloader):.4f}"
            f"  Accuracy={100*correct/total:.2f}%"
        )

In [9]:
train(
    model,
    trainloader,
    testloader,
    epochs=5
)

Epoch 1/5  Loss=0.3451  Accuracy=95.05%
Epoch 2/5  Loss=0.1293  Accuracy=96.72%
Epoch 3/5  Loss=0.0831  Accuracy=97.59%
Epoch 4/5  Loss=0.0632  Accuracy=97.59%
Epoch 5/5  Loss=0.0467  Accuracy=97.78%


In [10]:
print("Allocated Memory :",
      torch.cuda.memory_allocated()/1024**2,
      "MB")

print("Reserved Memory :",
      torch.cuda.memory_reserved()/1024**2,
      "MB")

Allocated Memory : 19.83984375 MB
Reserved Memory : 26.0 MB
